# Eksperimen Hyperparameter Tuning

Notebook akan menguji beberapa komponen berikut:
1. pengaruh depth dan width
2. pengaruh fungsi aktivasi
3. pengaruh learning rate

Dari setiap komponen pengujian akan dibandingkan: 
1. Hasil akhir prediksinya
2. Grafik training loss dan validation loss tiap epoch setelah pelatihan
3. Distribusi bobot dan gradien bobot dari setiap layer pada model

In [182]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
import ffnn

In [183]:
SEED=34

In [184]:
df = pd.read_csv("../data/dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cgpa                      10000 non-null  float64
 1   backlogs                  10000 non-null  int64  
 2   college_tier              10000 non-null  str    
 3   country                   10000 non-null  str    
 4   university_ranking_band   10000 non-null  str    
 5   internship_count          10000 non-null  int64  
 6   aptitude_score            10000 non-null  float64
 7   communication_score       10000 non-null  float64
 8   specialization            10000 non-null  str    
 9   industry                  10000 non-null  str    
 10  internship_quality_score  10000 non-null  float64
 11  placement_status          10000 non-null  str    
dtypes: float64(4), int64(2), str(6)
memory usage: 937.6 KB


In [185]:
df.describe()

,cgpa,backlogs,internship_count,aptitude_score,communication_score,internship_quality_score
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,6.998290,1.248100,1.49930,69.877531,65.158600,5.021436
std,0.802606,1.149904,1.20289,14.700532,14.740446,1.505975
min,4.000000,0.000000,0.00000,30.000000,30.000000,1.000000
25%,6.461928,0.000000,1.00000,59.880399,55.112244,4.012656
50%,6.997924,1.000000,1.00000,70.097368,65.006484,5.017335
75%,7.536865,2.000000,2.00000,80.213934,75.277248,6.031400
max,10.000000,6.000000,5.00000,100.000000,100.000000,10.000000


In [186]:
df.shape

(10000, 12)

# Data Preprocessing

we will label encode college_tier and university_ranking_band since they have an order

In [187]:
college_tier_map = {"Tier 3":0, "Tier 2":1, "Tier 1":2}
df["college_tier"] = df["college_tier"].map(college_tier_map)
df["college_tier"].value_counts()

college_tier
1    3993
2    3034
0    2973
Name: count, dtype: int64

In [188]:
ranking_map = {"300+":0,"100-300":1,"Top 100":2}
df["university_ranking_band"] = df["university_ranking_band"].map(ranking_map)
df["university_ranking_band"].value_counts()

university_ranking_band
0    4075
1    3975
2    1950
Name: count, dtype: int64

Target encoding for placement_status

In [189]:
target_map = {"Placed":1,"Not Placed":0}
df["placement_status"]=df["placement_status"].map(target_map)
df["placement_status"].value_counts()

placement_status
1    6153
0    3847
Name: count, dtype: int64

In [190]:
df.head()

,cgpa,backlogs,college_tier,country,university_ranking_band,internship_count,aptitude_score,communication_score,specialization,industry,internship_quality_score,placement_status
0,7.397371,1,1,Canada,1,2,53.574150,64.177062,Data Science,Consulting,5.481450,1
1,6.889389,0,0,UK,0,1,60.687750,88.346052,Data Science,Consulting,4.625099,1
2,7.518151,0,2,UK,1,2,64.568750,69.493171,Cybersecurity,Healthcare,5.227939,1
3,8.218424,0,1,UK,1,3,73.461500,78.204854,AI/ML,Tech,5.150674,1
4,6.812677,1,1,USA,1,4,86.518121,44.680881,Data Science,Consulting,3.888824,1


In [191]:
categorical_col = df.select_dtypes(include=["object"]).columns
print(categorical_col)

Index(['country', 'specialization', 'industry'], dtype='str')


Index(['country', 'specialization', 'industry'], dtype='str')


/tmp/ipykernel_6467/1470084899.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_col = df.select_dtypes(include=["object"]).columns


In [192]:
num_cols = df.select_dtypes(exclude=["object"]).columns
print(num_cols)

Index(['cgpa', 'backlogs', 'college_tier', 'university_ranking_band',
       'internship_count', 'aptitude_score', 'communication_score',
       'internship_quality_score', 'placement_status'],
      dtype='str')


### Preprocessing Pipeline

In [193]:
transformed_df = df.copy()

In [194]:
ss = StandardScaler()
ss.set_output(transform="pandas")

transformed_num_cols = transformed_df.select_dtypes(exclude="object").columns.drop("placement_status")
print(transformed_num_cols)

transformed_num_df = ss.fit_transform(transformed_df[transformed_num_cols])
transformed_num_df.head()

Index(['cgpa', 'backlogs', 'college_tier', 'university_ranking_band',
       'internship_count', 'aptitude_score', 'communication_score',
       'internship_quality_score'],
      dtype='str')


,cgpa,backlogs,college_tier,university_ranking_band,internship_count,aptitude_score,communication_score,internship_quality_score
0,0.497257,-0.215768,-0.007871,0.284641,0.416268,-1.109089,-0.066591,0.305474
1,-0.135691,-1.085449,-1.298153,-1.054846,-0.415104,-0.625164,1.573128,-0.263189
2,0.647749,-1.085449,1.282411,0.284641,0.416268,-0.361147,0.294074,0.137129
3,1.520292,-1.085449,-0.007871,0.284641,1.247641,0.243811,0.885109,0.085821
4,-0.231274,-0.215768,-0.007871,0.284641,2.079014,1.132029,-1.389289,-0.752116


In [195]:
oh_cols = ["country","specialization","industry"]

In [196]:
onehot = OneHotEncoder(sparse_output=False)

one_hot_encoded = onehot.fit_transform(transformed_df[oh_cols])


one_hot_df = pd.DataFrame(one_hot_encoded, 
                          columns=onehot.get_feature_names_out(oh_cols))

transformed_encoded_df = pd.concat([transformed_num_df, one_hot_df], axis=1)

transformed_encoded_df.head()

,cgpa,backlogs,college_tier,university_ranking_band,internship_count,aptitude_score,communication_score,internship_quality_score,country_Canada,country_Germany,...,specialization_Cloud,specialization_Core CS,specialization_Cybersecurity,specialization_Data Science,industry_Consulting,industry_Finance,industry_Healthcare,industry_Manufacturing,industry_Other,industry_Tech
0,0.497257,-0.215768,-0.007871,0.284641,0.416268,-1.109089,-0.066591,0.305474,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,-0.135691,-1.085449,-1.298153,-1.054846,-0.415104,-0.625164,1.573128,-0.263189,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.647749,-1.085449,1.282411,0.284641,0.416268,-0.361147,0.294074,0.137129,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1.520292,-1.085449,-0.007871,0.284641,1.247641,0.243811,0.885109,0.085821,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.231274,-0.215768,-0.007871,0.284641,2.079014,1.132029,-1.389289,-0.752116,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


# Modelling Experiment

In [197]:
X = transformed_encoded_df
y = transformed_df["placement_status"].values

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y, random_state=SEED)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 24), (2000, 24), (8000,), (2000,))

### Control Model

The model's performance is compared with other model variation

In [198]:
control_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Relu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

In [199]:
control_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
control_preds = control_model.predict(X_test)
print(f1_score(y_test,control_preds))

Epoch 1/20 [==============================] - train_loss: 0.523513
Epoch 2/20 [==============================] - train_loss: 0.518047
Epoch 3/20 [==============================] - train_loss: 0.517074
Epoch 4/20 [==============================] - train_loss: 0.511583
Epoch 5/20 [==============================] - train_loss: 0.512687
Epoch 6/20 [==============================] - train_loss: 0.521906
Epoch 7/20 [==============================] - train_loss: 0.516736
Epoch 8/20 [==============================] - train_loss: 0.508469
Epoch 9/20 [==============================] - train_loss: 0.510665
Epoch 10/20 [==============================] - train_loss: 0.512875
Epoch 11/20 [==============================] - train_loss: 0.507456
Epoch 12/20 [==============================] - train_loss: 0.508283
Epoch 13/20 [==============================] - train_loss: 0.508122
Epoch 14/20 [==============================] - train_loss: 0.507156
Epoch 15/20 [==============================] - train_loss

### Three variation of width with the same depth like the control

Width variation: 2, 8, and 64

In [200]:
w1_model = ffnn.Model(
    layers=[ffnn.Linear(24, 2), ffnn.Relu(), ffnn.Linear(2, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

w2_model = ffnn.Model(
    layers=[ffnn.Linear(24, 8), ffnn.Relu(), ffnn.Linear(8, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

w3_model = ffnn.Model(
    layers=[ffnn.Linear(24, 64), ffnn.Relu(), ffnn.Linear(64, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

In [201]:
w1_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
w1_preds = w1_model.predict(X_test)
print(f"width = 2: {f1_score(y_test, w1_preds)}")

Epoch 1/20 [=.............................] 10/250

Epoch 1/20 [=.............................] 10/250

Epoch 1/20 [==============================] - train_loss: 0.524480
Epoch 2/20 [==============================] - train_loss: 0.519874
Epoch 3/20 [==============================] - train_loss: 0.522134
Epoch 4/20 [==============================] - train_loss: 0.515263
Epoch 5/20 [==============================] - train_loss: 0.517342
Epoch 6/20 [==============================] - train_loss: 0.528317
Epoch 7/20 [==============================] - train_loss: 0.519789
Epoch 8/20 [==============================] - train_loss: 0.511913
Epoch 9/20 [==============================] - train_loss: 0.515186
Epoch 10/20 [==============================] - train_loss: 0.515210
Epoch 11/20 [==============================] - train_loss: 0.515844
Epoch 12/20 [==============================] - train_loss: 0.513781
Epoch 13/20 [==============================] - train_loss: 0.511384
Epoch 14/20 [==============================] - train_loss: 0.511589
Epoch 15/20 [==============================] - train_loss

In [202]:
w2_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
w2_preds = w2_model.predict(X_test)
print(f"width = 8: {f1_score(y_test, w2_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.525854
Epoch 2/20 [==============================] - train_loss: 0.520857
Epoch 3/20 [==============================] - train_loss: 0.520775
Epoch 4/20 [==============================] - train_loss: 0.514218
Epoch 5/20 [==============================] - train_loss: 0.515421
Epoch 6/20 [==============================] - train_loss: 0.525504
Epoch 7/20 [==============================] - train_loss: 0.517851
Epoch 8/20 [==============================] - train_loss: 0.509391
Epoch 9/20 [==============================] - train_loss: 0.512207
Epoch 10/20 [==============================] - train_loss: 0.513996
Epoch 11/20 [==============================] - train_loss: 0.509446
Epoch 12/20 [==============================] - train_loss: 0.509013
Epoch 13/20 [==============================] - train_loss: 0.509310
Epoch 14/20 [==============================] - train_loss: 0.508118
Epoch 15/20 [==============================] - train_loss

In [203]:
w3_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
w3_preds = w3_model.predict(X_test)
print(f"width = 64: {f1_score(y_test, w3_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.534365
Epoch 2/20 [==============================] - train_loss: 0.522957
Epoch 3/20 [==============================] - train_loss: 0.521232
Epoch 4/20 [==============================] - train_loss: 0.513834
Epoch 5/20 [==============================] - train_loss: 0.514486
Epoch 6/20 [==============================] - train_loss: 0.525260
Epoch 7/20 [==============================] - train_loss: 0.516974
Epoch 8/20 [==============================] - train_loss: 0.508443
Epoch 9/20 [==============================] - train_loss: 0.510085
Epoch 10/20 [==============================] - train_loss: 0.512398
Epoch 11/20 [==============================] - train_loss: 0.507703
Epoch 12/20 [==============================] - train_loss: 0.507935
Epoch 13/20 [==============================] - train_loss: 0.507823
Epoch 14/20 [==============================] - train_loss: 0.506896
Epoch 15/20 [==============================] - train_loss

#### Perbandingan

### Three variation of depth with the same width like the control

Depth variation (hidden layers): 
- 2
- 4
- 8


In [204]:
h1_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), 
            ffnn.Relu(), 
            ffnn.Linear(4, 4), 
            ffnn.Relu(),
            ffnn.Linear(4, 3), 
            ffnn.Softmax()
            ],
    loss=ffnn.CrossEntropyLoss(),
)

h2_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), 
            ffnn.Relu(), 
            ffnn.Linear(4, 4), 
            ffnn.Relu(), 
            ffnn.Linear(4, 4), 
            ffnn.Relu(),
            ffnn.Linear(4, 4), 
            ffnn.Relu(),
            ffnn.Linear(4, 3), 
            ffnn.Softmax()
            ],
    loss=ffnn.CrossEntropyLoss(),
)

h3_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), 
            ffnn.Relu(), 
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 4),
            ffnn.Relu(),
            ffnn.Linear(4, 3), 
            ffnn.Softmax()
            ],
    loss=ffnn.CrossEntropyLoss(),
)

In [205]:
h1_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
h1_preds = h1_model.predict(X_test)
print(f"hidden layer depth = 2: {f1_score(y_test,h1_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.530062
Epoch 2/20 [==============================] - train_loss: 0.520813
Epoch 3/20 [==============================] - train_loss: 0.523112
Epoch 4/20 [==============================] - train_loss: 0.513697
Epoch 5/20 [==============================] - train_loss: 0.517768
Epoch 6/20 [==============================] - train_loss: 0.521884
Epoch 7/20 [==============================] - train_loss: 0.519805
Epoch 8/20 [==============================] - train_loss: 0.512560
Epoch 9/20 [==============================] - train_loss: 0.514064
Epoch 10/20 [==============================] - train_loss: 0.517925
Epoch 11/20 [==============================] - train_loss: 0.519100
Epoch 12/20 [==============================] - train_loss: 0.514728
Epoch 13/20 [==============================] - train_loss: 0.516791
Epoch 14/20 [==============================] - train_loss: 0.511409
Epoch 15/20 [==============================] - train_loss

In [206]:
h2_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
h2_preds = h2_model.predict(X_test)
print(f"hidden layer depth = 4: {f1_score(y_test,h2_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.541685
Epoch 2/20 [==============================] - train_loss: 0.528522
Epoch 3/20 [==============================] - train_loss: 0.571054
Epoch 4/20 [==============================] - train_loss: 0.529460
Epoch 5/20 [==============================] - train_loss: 0.526525
Epoch 6/20 [==============================] - train_loss: 0.534618
Epoch 7/20 [==============================] - train_loss: 0.533338
Epoch 8/20 [==============================] - train_loss: 0.527851
Epoch 9/20 [==============================] - train_loss: 0.523490
Epoch 10/20 [==============================] - train_loss: 0.537108
Epoch 11/20 [==============================] - train_loss: 0.575540
Epoch 12/20 [==============================] - train_loss: 0.541520
Epoch 13/20 [==============================] - train_loss: 0.524270
Epoch 14/20 [==============================] - train_loss: 0.518124
Epoch 15/20 [==============================] - train_loss

In [207]:
h3_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
h3_preds = h3_model.predict(X_test)
print(f"hidden layer depth = 8: {f1_score(y_test,h3_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.677392
Epoch 2/20 [==============================] - train_loss: 0.546426
Epoch 3/20 [==============================] - train_loss: 0.544447
Epoch 4/20 [==============================] - train_loss: 0.530312
Epoch 5/20 [==============================] - train_loss: 0.531462
Epoch 6/20 [==============================] - train_loss: 0.538191
Epoch 7/20 [==============================] - train_loss: 0.544734
Epoch 8/20 [==============================] - train_loss: 0.537078
Epoch 9/20 [==============================] - train_loss: 0.524511
Epoch 10/20 [==============================] - train_loss: 0.535297
Epoch 11/20 [==============================] - train_loss: 0.532695
Epoch 12/20 [==============================] - train_loss: 0.606417
Epoch 13/20 [==============================] - train_loss: 0.575229
Epoch 14/20 [==============================] - train_loss: 0.528136
Epoch 15/20 [==============================] - train_loss

#### Perbandingan

### All variation of Activation Layer

FFNN is structured to have one of each input, hidden and output layer

Hidden layer variations are:

- RELU
- Leaky RELU
- ELU
- Sigmoid
- Tanh
- RMS Norm

In [208]:
relu_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Relu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

leaky_relu_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.LeakyRelu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

elu_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.ELU(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

sigmoid_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Sigmoid(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

tanh_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Tanh(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

rmsnorm_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.RMSNorm(4), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)




In [209]:
relu_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
relu_preds = relu_model.predict(X_test)
print(f"hidden layer activation = RELU: {f1_score(y_test,relu_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.528702
Epoch 2/20 [==============================] - train_loss: 0.520492
Epoch 3/20 [==============================] - train_loss: 0.530570
Epoch 4/20 [==============================] - train_loss: 0.516999
Epoch 5/20 [==============================] - train_loss: 0.517749
Epoch 6/20 [==============================] - train_loss: 0.524579
Epoch 7/20 [==============================] - train_loss: 0.522554
Epoch 8/20 [==============================] - train_loss: 0.515643
Epoch 9/20 [==============================] - train_loss: 0.514572
Epoch 10/20 [==============================] - train_loss: 0.522348
Epoch 11/20 [==============================] - train_loss: 0.539723
Epoch 12/20 [==============================] - train_loss: 0.518673
Epoch 13/20 [==============================] - train_loss: 0.512778
Epoch 14/20 [==============================] - train_loss: 0.512128
Epoch 15/20 [==============================] - train_loss

In [210]:
leaky_relu_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
leaky_relu_preds = leaky_relu_model.predict(X_test)
print(f"hidden layer activation = Leaky RELU: {f1_score(y_test, leaky_relu_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.526984
Epoch 2/20 [==============================] - train_loss: 0.518681
Epoch 3/20 [==============================] - train_loss: 0.517982
Epoch 4/20 [==============================] - train_loss: 0.512577
Epoch 5/20 [==============================] - train_loss: 0.514722
Epoch 6/20 [==============================] - train_loss: 0.521781
Epoch 7/20 [==============================] - train_loss: 0.517804
Epoch 8/20 [==============================] - train_loss: 0.511002
Epoch 9/20 [==============================] - train_loss: 0.512535
Epoch 10/20 [==============================] - train_loss: 0.515312
Epoch 11/20 [==============================] - train_loss: 0.511919
Epoch 12/20 [==============================] - train_loss: 0.510281
Epoch 13/20 [==============================] - train_loss: 0.511287
Epoch 14/20 [==============================] - train_loss: 0.509380
Epoch 15/20 [==============================] - train_loss

In [211]:
elu_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
elu_preds = elu_model.predict(X_test)
print(f"hidden layer activation = ELU: {f1_score(y_test, elu_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.527535
Epoch 2/20 [==============================] - train_loss: 0.518968
Epoch 3/20 [==============================] - train_loss: 0.518461
Epoch 4/20 [==============================] - train_loss: 0.512543
Epoch 5/20 [==============================] - train_loss: 0.512538
Epoch 6/20 [==============================] - train_loss: 0.523288
Epoch 7/20 [==============================] - train_loss: 0.515332
Epoch 8/20 [==============================] - train_loss: 0.507900
Epoch 9/20 [==============================] - train_loss: 0.510665
Epoch 10/20 [==============================] - train_loss: 0.510322
Epoch 11/20 [==============================] - train_loss: 0.505883
Epoch 12/20 [==============================] - train_loss: 0.509597
Epoch 13/20 [==============================] - train_loss: 0.504841
Epoch 14/20 [==============================] - train_loss: 0.506631
Epoch 15/20 [==============================] - train_loss

In [212]:
rmsnorm_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
rmsnorm_preds = rmsnorm_model.predict(X_test)
print(f"hidden layer activation = RMS Norm: {f1_score(y_test, rmsnorm_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.531544
Epoch 2/20 [==============================] - train_loss: 0.524580
Epoch 3/20 [==============================] - train_loss: 0.526296
Epoch 4/20 [==============================] - train_loss: 0.518881
Epoch 5/20 [==============================] - train_loss: 0.519002
Epoch 6/20 [==============================] - train_loss: 0.529613
Epoch 7/20 [==============================] - train_loss: 0.520926
Epoch 8/20 [==============================] - train_loss: 0.512542
Epoch 9/20 [==============================] - train_loss: 0.515451
Epoch 10/20 [==============================] - train_loss: 0.513481
Epoch 11/20 [==============================] - train_loss: 0.510310
Epoch 12/20 [==============================] - train_loss: 0.518626
Epoch 13/20 [==============================] - train_loss: 0.506470
Epoch 14/20 [==============================] - train_loss: 0.509662
Epoch 15/20 [==============================] - train_loss

In [213]:
tanh_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
tanh_preds = tanh_model.predict(X_test)
print(f"hidden layer activation = Tanh: {f1_score(y_test, tanh_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.531037
Epoch 2/20 [==============================] - train_loss: 0.526142
Epoch 3/20 [==============================] - train_loss: 0.525226
Epoch 4/20 [==============================] - train_loss: 0.519049
Epoch 5/20 [==============================] - train_loss: 0.519143
Epoch 6/20 [==============================] - train_loss: 0.527649
Epoch 7/20 [==============================] - train_loss: 0.520718
Epoch 8/20 [==============================] - train_loss: 0.512276
Epoch 9/20 [==============================] - train_loss: 0.514826
Epoch 10/20 [==============================] - train_loss: 0.515072
Epoch 11/20 [==============================] - train_loss: 0.510524
Epoch 12/20 [==============================] - train_loss: 0.510869
Epoch 13/20 [==============================] - train_loss: 0.508717
Epoch 14/20 [==============================] - train_loss: 0.510111
Epoch 15/20 [==============================] - train_loss

In [214]:
sigmoid_model.fit(X_train, y_train, epochs=20, lr=1, penalty="l2", lambda_=0.001)
sigmoid_preds = sigmoid_model.predict(X_test)
print(f"hidden layer activation = Sigmoid: {f1_score(y_test, sigmoid_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.527424
Epoch 2/20 [==============================] - train_loss: 0.523633
Epoch 3/20 [==============================] - train_loss: 0.521286
Epoch 4/20 [==============================] - train_loss: 0.517050
Epoch 5/20 [==============================] - train_loss: 0.518685
Epoch 6/20 [==============================] - train_loss: 0.523928
Epoch 7/20 [==============================] - train_loss: 0.518023
Epoch 8/20 [==============================] - train_loss: 0.512527
Epoch 9/20 [==============================] - train_loss: 0.514132
Epoch 10/20 [==============================] - train_loss: 0.514685
Epoch 11/20 [==============================] - train_loss: 0.513374
Epoch 12/20 [==============================] - train_loss: 0.509652
Epoch 13/20 [==============================] - train_loss: 0.510981
Epoch 14/20 [==============================] - train_loss: 0.510889
Epoch 15/20 [==============================] - train_loss

#### Perbandingan

### Three variation of Learning Rate

Using the same control model, we will modify the learning rate when fitting train data

Variation of learning rates:

- 0.1
- 10
- 40

In [215]:
lr1_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Relu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

lr2_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Relu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

lr3_model = ffnn.Model(
    layers=[ffnn.Linear(24, 4), ffnn.Relu(), ffnn.Linear(4, 3), ffnn.Softmax()],
    loss=ffnn.CrossEntropyLoss(),
)

In [216]:
lr1_model.fit(X_train, y_train, epochs=20, lr=0.1, penalty="l2", lambda_=0.001)
lr1_preds = lr1_model.predict(X_test)
print(f"Learning Rate = 0.1: {f1_score(y_test,lr1_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.567548
Epoch 2/20 [==============================] - train_loss: 0.536909
Epoch 3/20 [==============================] - train_loss: 0.527260
Epoch 4/20 [==============================] - train_loss: 0.520868
Epoch 5/20 [==============================] - train_loss: 0.518141
Epoch 6/20 [==============================] - train_loss: 0.517207
Epoch 7/20 [==============================] - train_loss: 0.515669
Epoch 8/20 [==============================] - train_loss: 0.513062
Epoch 9/20 [==============================] - train_loss: 0.513130
Epoch 10/20 [==============================] - train_loss: 0.512486
Epoch 11/20 [==============================] - train_loss: 0.511599
Epoch 12/20 [==============================] - train_loss: 0.512062
Epoch 13/20 [==============================] - train_loss: 0.511030
Epoch 14/20 [==============================] - train_loss: 0.511126
Epoch 15/20 [==============================] - train_loss

In [217]:
lr2_model.fit(X_train, y_train, epochs=20, lr=0.1, penalty="l2", lambda_=0.001)
lr2_preds = lr2_model.predict(X_test)
print(f"Learning Rate = 10: {f1_score(y_test,lr2_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.608809
Epoch 2/20 [==============================] - train_loss: 0.552243
Epoch 3/20 [==============================] - train_loss: 0.535551
Epoch 4/20 [==============================] - train_loss: 0.526051
Epoch 5/20 [==============================] - train_loss: 0.521350
Epoch 6/20 [==============================] - train_loss: 0.518848
Epoch 7/20 [==============================] - train_loss: 0.516699
Epoch 8/20 [==============================] - train_loss: 0.513464
Epoch 9/20 [==============================] - train_loss: 0.513835
Epoch 10/20 [==============================] - train_loss: 0.512562
Epoch 11/20 [==============================] - train_loss: 0.511682
Epoch 12/20 [==============================] - train_loss: 0.511509
Epoch 13/20 [==============================] - train_loss: 0.510377
Epoch 14/20 [==============================] - train_loss: 0.510529
Epoch 15/20 [==============================] - train_loss

In [218]:
lr3_model.fit(X_train, y_train, epochs=20, lr=0.1, penalty="l2", lambda_=0.001)
lr3_preds = lr3_model.predict(X_test)
print(f"Learning Rate = 40: {f1_score(y_test,lr3_preds)}")

Epoch 1/20 [==============================] - train_loss: 0.593981
Epoch 2/20 [==============================] - train_loss: 0.553805
Epoch 3/20 [==============================] - train_loss: 0.538883
Epoch 4/20 [==============================] - train_loss: 0.529737
Epoch 5/20 [==============================] - train_loss: 0.524669
Epoch 6/20 [==============================] - train_loss: 0.522526
Epoch 7/20 [==============================] - train_loss: 0.519430
Epoch 8/20 [==============================] - train_loss: 0.515886
Epoch 9/20 [==============================] - train_loss: 0.515735
Epoch 10/20 [==============================] - train_loss: 0.514547
Epoch 11/20 [==============================] - train_loss: 0.513189
Epoch 12/20 [==============================] - train_loss: 0.513240
Epoch 13/20 [==============================] - train_loss: 0.512011
Epoch 14/20 [==============================] - train_loss: 0.512198
Epoch 15/20 [==============================] - train_loss

#### Perbandingan